In [1]:
import tensorflow as tf
import numpy as np

import matplotlib.pyplot as plt


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
datadir = "/content/drive/MyDrive/Colab Notebooks/SolarPanelDefectClassification/Data"
img_height = 224
img_weight = 224
batch_size = 32
seed = 42

In [4]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    datadir,
    validation_split = 0.2,
    subset = "training",
    image_size = (img_height, img_weight),
    batch_size = batch_size,
    seed = seed,
)

Found 871 files belonging to 6 classes.
Using 697 files for training.


In [5]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    datadir,
    validation_split = 0.2,
    subset = "validation",
    image_size = (img_height, img_weight),
    batch_size = batch_size,
    seed = seed,
)

Found 871 files belonging to 6 classes.
Using 174 files for validation.


In [6]:
class_names = train_dataset.class_names
num_classes = len(class_names)
print(class_names)

['Bird-drop', 'Clean', 'Dusty', 'Electrical-damage', 'Physical-Damage', 'Snow-Covered']


## Transfer Learning - EfficientNetB0

In [14]:
class_counts = {}
total_images = 0

for class_name in class_names:
  class_path = os.path.join(datadir, class_name)
  count = len(os.listdir(class_path))
  class_counts[class_name] = count
  total_images += count

print("Class Count: ", class_counts)

Class Count:  {'Bird-drop': 191, 'Clean': 193, 'Dusty': 192, 'Electrical-damage': 103, 'Physical-Damage': 69, 'Snow-Covered': 123}


In [15]:
class_weights = {}

for index, class_name in enumerate(class_names):
  class_weights[index] = total_images / (num_classes * class_counts[class_name])

print("Class weights: ", class_weights)

Class weights:  {0: 0.7600349040139616, 1: 0.7521588946459413, 2: 0.7560763888888888, 3: 1.4093851132686084, 4: 2.103864734299517, 5: 1.1802168021680217}


In [17]:
train_images = []
train_labels = []

for images, labels in train_dataset:
  images = preprocess_input(images)
  train_images.append(images)
  train_labels.append(labels)

In [18]:
train_images = tf.concat(train_images, axis=0)
train_labels = tf.concat(train_labels, axis=0)

In [19]:
val_images = []
val_labels = []

for images, labels in  validation_dataset:
  images = preprocess_input(images)
  val_images.append(images)
  val_labels.append(labels)

val_images = tf.concat(val_images, axis=0)
val_labels = tf.concat(val_labels, axis=0)

## Applying HPO on EfficientNetB07

In [ ]:

import keras_tuner as kt
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

In [ ]:

# Reuse your train_dataset and validation_dataset (with preprocess_input already applied)
# Make sure they are defined as in your original code

def build_model(hp):
    base_model = EfficientNetB7(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    model = Sequential([
        RandomFlip("horizontal"),
        RandomRotation(hp.Float("rotation_factor", min_value=0.05, max_value=0.3, step=0.05)),
        RandomZoom(hp.Float("zoom_factor", min_value=0.05, max_value=0.3, step=0.05)),
        base_model,
        GlobalAveragePooling2D(),
        Dropout(hp.Float("dropout_rate", min_value=0.0, max_value=0.5, step=0.1)),
        Dense(
            hp.Int("dense_units", min_value=64, max_value=512, step=64),
            activation='relu'
        ),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:


# Setup Random Search Tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,          # Same as n_iter before
    executions_per_trial=1,
    directory='kt_dir',
    project_name='efficientnet_tune',
    overwrite=True
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

In [ ]:


# Run the search
tuner.search(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=[early_stopping],
    class_weight=class_weights  # Handles imbalance
)

# Results
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters:")
print(best_hps.values)

best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate on validation
val_loss, val_acc = best_model.evaluate(validation_dataset)
print(f"Best validation accuracy: {val_acc:.4f}")

Trial 20 Complete [00h 05m 10s]
val_accuracy: 0.7873563170433044

Best val_accuracy So Far: 0.8218390941619873
Total elapsed time: 01h 20m 00s
Best hyperparameters:
{'rotation_factor': 0.05, 'zoom_factor': 0.05, 'dropout_rate': 0.2, 'dense_units': 320, 'learning_rate': 0.0009080065553439061}


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 477ms/step - accuracy: 0.8218 - loss: 0.6405
Best validation accuracy: 0.8218


## Classification Report

In [28]:
from sklearn.metrics import f1_score, classification_report
import numpy as np

# Get predictions from the best model on the validation set
val_predictions = best_model.predict(val_images)
predicted_classes = np.argmax(val_predictions, axis=1)

# Calculate F1-score
f1 = f1_score(val_labels, predicted_classes, average='weighted')
print(f"F1 Score: {f1:.4f}")

# Display classification report for more detailed metrics
print("\nClassification Report:")
print(classification_report(val_labels, predicted_classes, target_names=class_names))

6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 558ms/step
F1 Score: 0.8220

Classification Report:
                   precision    recall  f1-score   support

        Bird-drop       0.87      0.82      0.85        40
            Clean       0.65      0.97      0.78        33
            Dusty       0.87      0.69      0.77        48
Electrical-damage       0.81      1.00      0.90        13
  Physical-Damage       1.00      0.70      0.82        20
     Snow-Covered       0.95      0.90      0.92        20

         accuracy                           0.82       174
        macro avg       0.86      0.85      0.84       174
     weighted avg       0.85      0.82      0.82       174



In [29]:
best_model.save(f"/content/drive/MyDrive/Colab Notebooks/Solar Panel Classification/Model/effnet_{val_acc:.2f}.h5")